# Notebook 03 — Clustering & Recommendation

**Objective:** Apply PCA + K-Means to the processed feature matrix to
discover natural groupings in professional profiles, evaluate cluster
quality, interpret what each cluster represents, and demonstrate
profile recommendations using cosine similarity.

**Inputs:**
- `data/processed/X_processed.npy` — feature matrix from notebook 02
- `data/raw/profiles.csv` — original profiles for display layer

**Outputs:**
- `data/processed/cluster_labels.npy` — cluster assignment per profile
- `outputs/plots/` — PCA variance, elbow, silhouette, t-SNE figures
- `outputs/clusters/` — one CSV summary per cluster

In [ ]:
import os
import sys

NOTEBOOK_DIR = os.path.abspath("")
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import davies_bouldin_score, silhouette_score

from src.config import (
    RAW_DATA_PATH, PROCESSED_PATH, PLOTS_DIR,
    PCA_VARIANCE, K_RANGE
)
from src.models.clustering import (
    run_kmeans_experiment,
    plot_elbow_and_silhouette,
    fit_final_kmeans,
    profile_clusters
)
from src.models.compatibility import recommend_profiles

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["savefig.bbox"] = "tight"
warnings.filterwarnings("ignore")
print("✓ Imports complete")

In [ ]:
# Feature matrix produced by notebook 02
X = np.load(PROCESSED_PATH)
print(f"Feature matrix shape : {X.shape}")

# Full profiles for display — never fed into the model
df_original = pd.read_csv(RAW_DATA_PATH)
print(f"Profiles loaded      : {len(df_original):,}")

# Load skill vocabulary for cluster interpretation
vocab_path = os.path.join(PROJECT_ROOT, "data", "processed", "skill_vocab.txt")
with open(vocab_path) as f:
    skill_vocab = [line.strip() for line in f.readlines()]
print(f"Skill vocabulary     : {len(skill_vocab)} terms")

## 1. Dimensionality Reduction — PCA

The feature matrix has ~200 dimensions. K-Means degrades in high
dimensions because distances become increasingly uniform (curse of
dimensionality). PCA compresses the matrix while retaining 95% of
variance, making cluster boundaries meaningful.

t-SNE will be used separately for 2D visualisation only —
clustering is always performed on PCA output, never on t-SNE output.
t-SNE distorts global structure and is not suitable as a clustering input.

In [ ]:
pca = PCA(n_components=PCA_VARIANCE, random_state=42)
X_pca = pca.fit_transform(X)

print(f"Original dimensions  : {X.shape[1]}")
print(f"PCA dimensions       : {X_pca.shape[1]}")
print(f"Variance retained    : {pca.explained_variance_ratio_.sum()*100:.2f}%")
print(f"Compression ratio    : {X.shape[1]/X_pca.shape[1]:.1f}x")

In [ ]:
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Per-component variance
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_ * 100,
            color="#4A90D9", edgecolor="white")
axes[0].set_title("Variance explained per component",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Variance explained (%)")

# Cumulative variance with 95% threshold line
axes[1].plot(range(1, len(cumulative_variance) + 1),
             cumulative_variance * 100,
             marker=".", color="#7B68EE", linewidth=2)
axes[1].axhline(y=95, color="#E8834E", linestyle="--",
                linewidth=1.5, label="95% threshold")
axes[1].axvline(x=X_pca.shape[1], color="#5CB85C", linestyle="--",
                linewidth=1.5,
                label=f"Selected: {X_pca.shape[1]} components")
axes[1].set_title("Cumulative variance explained",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance (%)")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "09_pca_explained_variance.png"))
plt.show()

## 2. Choosing K — Elbow Method + Silhouette Score

Two complementary metrics are used to select K:

- **Inertia (elbow curve):** sum of squared distances from each point
  to its cluster centre. Lower is better, but always decreases with K.
  The "elbow" — where the rate of improvement slows — suggests a good K.

- **Silhouette score:** measures how similar a profile is to its own
  cluster versus neighbouring clusters. Ranges from -1 to 1.
  Higher is better. The peak identifies the most well-separated K.

Using both together avoids the weakness of either metric alone.

In [ ]:
results_df = run_kmeans_experiment(X_pca)
plot_elbow_and_silhouette(results_df)

In [ ]:
# Read the plots above and set BEST_K to your chosen value
# The silhouette peak is the primary signal
# The elbow confirms it — look for where inertia stops dropping sharply

BEST_K = int(results_df.loc[results_df["silhouette"].idxmax(), "k"])

print(f"Selected K : {BEST_K}")
print(f"\nFull results table:")
print(results_df.to_string(index=False))

## 3. Fit Final Model

In [ ]:
labels = fit_final_kmeans(X_pca, BEST_K)

# Write labels back to df_original — this is the bridge between
# the model output and the human-readable display layer
df_original["cluster"] = labels

# Save labels for later use
labels_path = os.path.join(PROJECT_ROOT, "data", "processed",
                           "cluster_labels.npy")
np.save(labels_path, labels)
print(f"✓ Cluster labels saved → {labels_path}")

In [ ]:
sil_score = silhouette_score(X_pca, labels, sample_size=5000,
                             random_state=42)
db_score  = davies_bouldin_score(X_pca, labels)

print("=" * 40)
print("Final model evaluation")
print("=" * 40)
print(f"K (clusters)           : {BEST_K}")
print(f"Silhouette score       : {sil_score:.4f}  (higher better, max 1.0)")
print(f"Davies-Bouldin index   : {db_score:.4f}  (lower better, min 0.0)")
print("=" * 40)

## 4. Cluster Interpretation

For each cluster we compute the top skills, dominant industry, dominant
role, median experience, and seniority breakdown. This is what allows
clusters to be given human-readable names — the most important output
for interpretability.

In [ ]:
summary_df = profile_clusters(labels, df_original, skill_vocab)

print("Cluster summary:")
print(summary_df.to_string(index=False))

# Save one CSV per cluster into outputs/clusters/
clusters_dir = os.path.join(PROJECT_ROOT, "outputs", "clusters")
for cluster_id in sorted(df_original["cluster"].unique()):
    subset = df_original[df_original["cluster"] == cluster_id]
    out_path = os.path.join(clusters_dir,
                            f"cluster_{cluster_id}_profiles.csv")
    subset.to_csv(out_path, index=False)

print(f"\nCluster CSVs saved → {clusters_dir}")

Cluster summary:
 cluster  size  pct_of_total                                top_skills dominant_industry   dominant_role  median_exp top_seniority
       0 25114          50.2    c, microservices, invision, numpy, php        Consulting       Assistant         2.0         entry
       1 24886          49.8 c, rust, data science, cicd, google cloud            Energy Senior Engineer        15.0        senior

✓ Cluster CSVs saved → c:\programmes\unsupervised-professional-matching\outputs\clusters


### Cluster names (based on profiling output above)

| Cluster | Size | Interpretation |
|---|---|---|
| 0 | X | e.g. "Early-career engineers in fintech" |
| 1 | X | e.g. "Senior data professionals in healthcare" |
| 2 | X | ... |

*(Fill this in after running cell 15 — this is the most important
interpretability output of the entire project)*`

## 5. t-SNE Visualisation

t-SNE projects the high-dimensional PCA space down to 2D for visual
inspection. Profiles that are close together in 2D were also close in
the original feature space.

Important: t-SNE is used for visualisation only. Clustering was
performed on PCA output — never on t-SNE coordinates, which distort
global distances and are not suitable for K-Means input.

In [ ]:
print("Running t-SNE — this takes 2-4 minutes on 50k profiles...")

tsne = TSNE(n_components=2, random_state=42, perplexity=40,
            max_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_pca)

# Build a colour palette — one distinct colour per cluster
palette = sns.color_palette("tab10", n_colors=BEST_K)

fig, ax = plt.subplots(figsize=(12, 8))

for cluster_id in range(BEST_K):
    mask = labels == cluster_id
    ax.scatter(
        X_tsne[mask, 0], X_tsne[mask, 1],
        c=[palette[cluster_id]],
        label=f"Cluster {cluster_id} (n={mask.sum():,})",
        alpha=0.4, s=4, linewidths=0
    )

ax.set_title("t-SNE projection coloured by cluster",
             fontsize=14, fontweight="bold")
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.legend(markerscale=3, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "10_tsne_clusters.png"))
plt.show()

## 6. Recommendation Demo

This is the practical output of the system — given any profile, surface
the most similar professionals within the same cluster.

The two-stage process:
1. Cluster membership narrows the search from 50,000 to ~cluster size
2. Cosine similarity ranks within the cluster by feature vector similarity

The model only ever operated on anonymous row indices.
Full profile details are retrieved from df_original at display time.

In [ ]:
# Pick any index to use as the query profile
# Try a few different ones to show the system working across clusters
QUERY_IDX = 0

query = df_original.iloc[QUERY_IDX]
print("=" * 55)
print("QUERY PROFILE")
print("=" * 55)
print(f"Name            : {query['name']}")
print(f"Role            : {query['current_role']}")
print(f"Company         : {query['current_company']}")
print(f"Industry        : {query['industry']}")
print(f"Seniority       : {query['seniority_level']}")
print(f"Experience      : {query['years_experience']} years")
print(f"Location        : {query['location']}")
print(f"Remote pref     : {query['remote_preference']}")
print(f"Skills          : {query['skills']}")
print(f"Goals           : {query['goals']}")
print(f"Needs           : {query['needs']}")
print(f"Can offer       : {query['can_offer']}")
print(f"Cluster         : {query['cluster']}")

df_original.head(1)

In [ ]:
recommendations = recommend_profiles(
    query_idx   = QUERY_IDX,
    X           = X,            # full feature matrix — not PCA reduced
    labels      = labels,
    df_original = df_original,
    top_n       = 5
)

print("=" * 55)
print(f"TOP 5 SIMILAR PROFILES  (Cluster {labels[QUERY_IDX]})")
print("=" * 55)

for rank, (_, row) in enumerate(recommendations.iterrows(), start=1):
    print(f"\nRank {rank}  —  similarity: {row['similarity_score']:.4f}")
    print(f"  Name        : {row['name']}")
    print(f"  Role        : {row['current_role']}")
    print(f"  Company     : {row['current_company']}")
    print(f"  Industry    : {row['industry']}")
    print(f"  Seniority   : {row['seniority_level']}")
    print(f"  Experience  : {row['years_experience']} years")
    print(f"  Location    : {row['location']}")
    print(f"  Skills      : {row['skills']}")
    print(f"  Goals       : {row['goals']}")
    print(f"  Needs       : {row['needs']}")
    print(f"  Can offer   : {row['can_offer']}")
    print("-" * 55)

## 7. Results Summary

| Metric | Value |
|---|---|
| Profiles clustered | 50,000 |
| Features in matrix | ~220 |
| PCA components retained | X (95% variance) |
| Optimal K | X |
| Silhouette score | X |
| Davies-Bouldin index | X |

### What each cluster represents
*(Copy your named cluster table from cell 16 here)*

### Key findings
- Cluster X is the largest, dominated by Y professionals in Z industry
- Skills most strongly associated with cluster separation: ...
- The recommendation engine surfaces profiles with similarity scores
  above X, indicating meaningful feature overlap

**Next (Week 16):** Compare K-Means against DBSCAN and Agglomerative
Clustering using the same metrics. Add `04_results.ipynb`.